# Regresión lineal con el dataset Pima (repaso)

**Idea clave:** la regresión lineal predice un **número** (aquí, el `BMI`). La regresión logística predice una **categoría** (0 o 1).

| | Lineal | Logística |
|---|---|---|
| Qué predice | Un valor continuo | Una clase (0 o 1) |
| Forma del modelo | Una recta | Una curva sigmoide |
| Métricas | R², MAE, RMSE | Matriz de confusión, precisión, recall, F1 |

## 1. Importar bibliotecas

In [ ]:
import pandas as pd  # manejo de tablas de datos
from sklearn.model_selection import train_test_split  # separa entrenamiento y prueba
from sklearn.linear_model import LinearRegression  # modelo de regresión lineal
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error  # métricas para números
import matplotlib.pyplot as plt  # gráficos

## 2. Leer los datos

In [ ]:
datos = pd.read_csv('diabetes.csv')  # lee el archivo; la primera fila son los nombres de columnas
datos.head()  # muestra las primeras 5 filas para confirmar que se leyó bien

## 3. Limpiar los ceros de BMI

Un `BMI` de 0 es un dato faltante, no un valor real. Quitamos esas filas.

In [ ]:
datos = datos[datos['BMI'] > 0]  # conserva solo las filas donde BMI es mayor que 0
print(datos.shape)  # cuántas filas quedaron

## 4. Separar X (características) y y (valor a predecir)

In [ ]:
X = datos.drop(columns=['BMI', 'Outcome'])  # quitamos BMI (la respuesta) y Outcome (otra salida, categórica)
y = datos['BMI']  # la respuesta ahora es un número continuo
print(X.shape, y.shape)  # X tiene 7 características; y tiene un valor por persona

## 5. Dividir en entrenamiento y prueba

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=16)  # 75 % entrenar, 25 % probar; random_state fija la división
print(X_train.shape, X_test.shape)  # tamaño de cada grupo

## 6. Entrenar el modelo lineal

In [ ]:
modelo = LinearRegression()  # crea el modelo; no necesita max_iter porque tiene solución directa
modelo.fit(X_train, y_train)  # aprende la recta: BMI = intercepto + coef1·x1 + ... + coef7·x7
y_pred = modelo.predict(X_test)  # predice el BMI de las personas del grupo de prueba

## 7. Coeficientes

Positivo: esa característica sube el BMI. Negativo: lo baja. Su tamaño depende de la escala de cada columna.

In [ ]:
coef = pd.Series(modelo.coef_, index=X.columns)  # une cada coeficiente con el nombre de su columna
print(coef)  # un peso por característica
print('Intercepto:', modelo.intercept_)  # valor base del modelo cuando todas las características valen 0

## 8. Métricas de error

- **R²:** qué parte de la variación del BMI explica el modelo. 1 = perfecto; 0 = no sirve para nada.
- **MAE:** error absoluto medio. En promedio, cuántos puntos de BMI se equivoca el modelo.
- **RMSE:** raíz del error cuadrático medio. Castiga más los errores grandes y está en la misma unidad que el BMI.

In [ ]:
r2 = r2_score(y_test, y_pred)  # R²: proporción de la variación explicada
mae = mean_absolute_error(y_test, y_pred)  # promedio del error absoluto
rmse = mean_squared_error(y_test, y_pred) ** 0.5  # raíz del promedio de errores al cuadrado
print(f'R² = {r2:.3f}')
print(f'MAE = {mae:.2f}')
print(f'RMSE = {rmse:.2f}')

**Cómo leerlo (ejemplo):** si sale `R² = 0.30`, el modelo explica el 30 % de la variación del BMI. Si sale `MAE = 4.5`, en promedio se equivoca por 4.5 puntos de BMI. Un R² bajo no es error del código: significa que las características del dataset explican poco el BMI.

## 9. Gráfico real vs predicho

In [ ]:
plt.scatter(y_test, y_pred, alpha=0.6)  # cada punto es una persona: X = BMI real, Y = BMI predicho
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red')  # línea roja: predicción perfecta
plt.xlabel('BMI real')  # etiqueta del eje X
plt.ylabel('BMI predicho')  # etiqueta del eje Y
plt.title('Regresión lineal: real vs predicho')  # título del gráfico
plt.show()  # muestra el gráfico

**Cómo leerlo:** mientras más cerca estén los puntos de la línea roja, mejor predice el modelo. Si los puntos se dispersan mucho, el modelo se equivoca más.